# BT4012 temporal graph-feature LightGBM ensemble

This notebook predicts transaction labels using three complementary feature views:

1. **All features**: the original tabular variables plus graph-derived variables.
2. **Tabular features**: the original non-graph variables only.
3. **Graph features**: historical network information such as PageRank, degree, and component size.

The workflow uses time-aware validation, selects blend weights from out-of-fold predictions, adds only high-confidence pseudo-labels, and creates the final Kaggle submission.

## Expected data layout

Place the competition files in a relative `data/` directory next to this notebook:

```text
data/
  train.csv
  test.csv
  txs_edgelist.csv
  sample_submission.csv
```

The dataset is intentionally not included in the repository. The setup instructions are included in this notebook.

## 0. Environment setup

Run the next cell once in a fresh notebook environment. It installs the pinned versions used for the final submission. The competition dataset is not installed or downloaded by this cell; place the four CSV files in the relative `data/` directory described below.

In [ ]:
# Install the pinned dependencies listed in requirements.txt.
%pip install -q -r requirements.txt

## 1. Imports and reproducibility

The final model uses LightGBM for tabular classification, NetworkX for historical graph statistics, pandas for data handling, NumPy for numerical operations, and scikit-learn for ROC-AUC evaluation. 

The `SEED` value controls the model seeds, while the explicit single-thread setting in the model cell controls reproducibility. The final code intentionally avoids a hidden dependency on a local Kaggle cache path. 

The final pipeline uses deterministic model seeds and one CPU thread; this is important because multithreaded LightGBM runs produced different predictions for some borderline transactions during experimentation.

In [3]:
# Import the modelling and graph-analysis libraries used by the final pipeline.
from pathlib import Path
import warnings

import lightgbm as lgb
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = Path("data")
OUT = Path("submission_improved_temporal_graph_ensemble.csv")

GRAPH_COLUMNS = [
    "graph_pagerank", "component_id", "component_size",
    "clustering_coeff", "k_core", "reciprocity", "log_in_degree",
    "log_out_degree", "in_degree", "out_degree", "total_degree",
    "flow_ratio", "log_component_size",
]

print(f"Data directory: {DATA_DIR}")
print(f"Output path: {OUT.resolve()}")

Data directory: data
Output path: C:\Users\regan\OneDrive\Desktop\Uni\Y4S1\BT4012\BT4012_Kaggle_Project\submission_improved_temporal_graph_ensemble.csv


## 2. Load and inspect the competition data

This section loads the four competition files from the relative `data/` directory:

- `train.csv`: transaction features and partially observed labels
- `test.csv`: future transactions requiring predictions
- `txs_edgelist.csv`: directed fund-flow relationships
- `sample_submission.csv`: the required output format

The inspection cell reports the dataset shapes, label distribution, and time-step ranges. These checks confirm that the files have been loaded correctly before graph construction and modelling begin.

The `label` column contains three types of information: genuine `0` labels for licit transactions, genuine `1` labels for illicit transactions, and unknown values. Unknown values are not treated as licit transactions and are not used as true supervised targets. They may be used later only for conservative pseudo-labelling.

The data is also time-ordered: the training transactions occur before the test transactions. This means the final validation strategy must imitate future prediction by training on earlier periods and validating on later periods, rather than randomly shuffling all rows.

In [4]:
# Load the competition inputs and the required submission template later.
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Edges shape: {edges.shape}")
print(f"Label counts:\n{train['label'].value_counts(dropna=False)}")
print(f"Train time steps: {train['time_step'].min()} to {train['time_step'].max()}")
print(f"Test time steps: {test['time_step'].min()} to {test['time_step'].max()}")

Train shape: (141772, 168)
Test shape: (15329, 168)
Edges shape: (234355, 2)
Label counts:
label
NaN    110537
0.0     27591
1.0      3644
Name: count, dtype: int64
Train time steps: 1 to 35
Test time steps: 36 to 49


## 3. Build historical graph features

For a transaction at time step `t`, only edges whose two endpoint transactions occurred before `t` are used. This avoids using future graph information. The features capture centrality, connectivity, local clustering, reciprocity, flow direction, and component structure.

`scale_pos_weight` increases the contribution of illicit examples during tree fitting. It changes the training objective, not the ROC-AUC definition, and is useful here because the labelled data contains 27,591 licit rows but only 3,644 illicit rows. The graph-only model uses fewer leaves to reduce overfitting because its future-node coverage is limited.

### Why these graph features?

- **PageRank** measures importance in directed fund-flow structure.
- **In/out degree** measures how many incoming and outgoing relationships a transaction has.
- **Component size** describes the scale of its connected transaction ecosystem.
- **Clustering coefficient** captures local network density.
- **K-core number** identifies nodes embedded in densely connected regions.
- **Reciprocity** measures bidirectional relationships.
- **Flow ratio** distinguishes receiving-heavy from sending-heavy nodes.

The graph is rebuilt for each time step. This is more expensive than using one graph over all rows, but it prevents future relationships from leaking into earlier observations. Transactions that do not yet appear in the historical graph receive zero-valued graph features.

In [5]:
# Build one leakage-aware historical graph snapshot per time step.
def add_historical_graph_features(train, test, edges):
    all_rows = pd.concat(
        [train[["txId", "time_step"]], test[["txId", "time_step"]]],
        ignore_index=True,
    ).drop_duplicates("txId")
    tx_time = all_rows.set_index("txId")["time_step"]
    source, target = edges.columns[:2]
    timed = edges.copy()
    timed["source_time"] = timed[source].map(tx_time)
    timed["target_time"] = timed[target].map(tx_time)
    timed = timed.dropna(subset=["source_time", "target_time"])

    train = train.copy()
    test = test.copy()
    for frame in (train, test):
        for column in GRAPH_COLUMNS:
            frame[column] = 0.0

    for step in sorted(pd.concat([train["time_step"], test["time_step"]]).unique()):
        historical = timed[
            (timed["source_time"] < step) & (timed["target_time"] < step)
        ]
        if historical.empty:
            continue
        graph = nx.from_pandas_edgelist(
            historical, source=source, target=target, create_using=nx.DiGraph
        )
        undirected = graph.to_undirected()
        pagerank = nx.pagerank(graph, alpha=0.85, max_iter=100)
        clustering = nx.clustering(undirected)
        core = nx.core_number(undirected) if undirected.number_of_edges() else {}
        component_id, component_size = {}, {}
        components = sorted(nx.connected_components(undirected), key=lambda c: min(c))
        for number, component in enumerate(components):
            for node in component:
                component_id[node] = number
                component_size[node] = len(component)

        values = {}
        for node in graph.nodes:
            outgoing = set(graph.successors(node))
            incoming = set(graph.predecessors(node))
            neighbours = outgoing | incoming
            in_degree = len(incoming)
            out_degree = len(outgoing)
            size = component_size.get(node, 0)
            values[node] = [
                pagerank.get(node, 0.0), component_id.get(node, -1),
                size, clustering.get(node, 0.0), core.get(node, 0),
                len(outgoing & incoming) / len(neighbours) if neighbours else 0.0,
                np.log1p(in_degree), np.log1p(out_degree), in_degree,
                out_degree, in_degree + out_degree,
                (in_degree + 1.0) / (out_degree + 1.0), np.log1p(size),
            ]
        for frame in (train, test):
            mask = frame["time_step"].eq(step)
            if mask.any():
                frame.loc[mask, GRAPH_COLUMNS] = (
                    frame.loc[mask, ["txId"]]["txId"]
                    .map(values)
                    .apply(lambda row: row if isinstance(row, list) else [0.0] * len(GRAPH_COLUMNS))
                    .tolist()
                )
    return train, test

train, test = add_historical_graph_features(train, test, edges)
print("Historical graph features added.")

Historical graph features added.


## 4. Prepare labels and feature views

Unknown labels are excluded from supervised fitting at this stage. Three feature views are created: all features, tabular-only features, and graph-only features. The arbitrary connected-component identifier is excluded.

Pseudo-labelling is used as a semi-supervised learning technique rather than as a replacement for true labels. The 0.98 confidence threshold is deliberately strict: adding many uncertain pseudo-labels could reinforce the model's mistakes. The pseudo-label source is restricted to early historical rows, so the model does not use future test-period information to manufacture training labels.

The original anonymised variables are retained because they contain transaction-level and precomputed neighbourhood information supplied by the competition. The three feature views allow us to test whether historical graph information adds value beyond the supplied tabular variables. The component identifier is excluded because its numeric value is an arbitrary label, not an ordered quantity.

In [6]:
# Keep only genuine 0/1 labels for supervised training; unknown labels remain missing.
train["label"] = pd.to_numeric(train["label"], errors="coerce")
labelled = train["label"].isin([0, 1])
features = [
    c for c in train.columns
    if c not in {"txId", "label", "index", "component_id"}
    and c in test.columns
]
graph_features = [c for c in GRAPH_COLUMNS if c in features and c != "component_id"]
tabular_features = [c for c in features if c not in graph_features]
sets = {
    "all": features,
    "tabular": tabular_features,
    "graph": graph_features,
}
y = train.loc[labelled, "label"].astype("int8")
x = train.loc[labelled]
pos_weight = float((y == 0).sum() / max((y == 1).sum(), 1))

print(f"Labelled rows: {len(y):,}")
print(f"Unknown rows: {(~labelled).sum():,}")
print(f"Feature counts: { {name: len(columns) for name, columns in sets.items()} }")
print(f"Positive class weight: {pos_weight:.4f}")

Labelled rows: 31,235
Unknown rows: 110,537
Feature counts: {'all': 178, 'tabular': 166, 'graph': 12}
Positive class weight: 7.5716


## 5. Define the LightGBM model

LightGBM is used because it models nonlinear feature interactions efficiently and handles the high-dimensional anonymised transaction features. The positive class is weighted to account for imbalance. A single CPU thread is intentional: it makes the final CSV reproducible across runs.

LightGBM was selected for its ability to model nonlinear interactions efficiently while handling many numeric features. The illicit class is rare, so `scale_pos_weight` increases the contribution of positive examples during training. The graph-only model uses fewer leaves to reduce overfitting because future transactions often have limited historical topology.

In [7]:
# Use one CPU thread so repeated runs produce the same LightGBM predictions.
def make_model(pos_weight, seed, leaves=31):
    return lgb.LGBMClassifier(
        objective="binary", n_estimators=600, learning_rate=0.03,
        num_leaves=leaves, min_child_samples=30, subsample=0.8,
        subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=seed, n_jobs=1, verbosity=-1,
    )

## 6. Chronological out-of-fold validation and blend selection

The three folds are expanding temporal folds. Models train on earlier labelled periods and validate on later periods. Blend weights are selected only from rows with complete out-of-fold predictions, preventing in-sample predictions from influencing the ensemble decision.

For each fold, all validation rows occur after the corresponding fitting rows. These are expanding temporal folds rather than independent random samples, so they expose possible temporal drift. Blend weights are selected only from complete out-of-fold predictions. The grid requires non-negative weights summing to one, keeping the final ensemble stable and interpretable.

In [8]:
# Generate out-of-fold predictions using chronological, expanding-window folds.
labelled_steps = np.sort(x["time_step"].unique())
fractions = [(0.50, 0.65), (0.65, 0.80), (0.80, 1.00)]
oof = pd.DataFrame(np.nan, index=x.index, columns=list(sets))
for fold, (start_fraction, end_fraction) in enumerate(fractions):
    start = labelled_steps[int(len(labelled_steps) * start_fraction)]
    end = labelled_steps[min(int(len(labelled_steps) * end_fraction), len(labelled_steps) - 1)]
    fit = x["time_step"] < start
    valid = (x["time_step"] >= start) & (x["time_step"] <= end)
    for number, (name, columns) in enumerate(sets.items()):
        model = make_model(pos_weight, 1000 + fold * 10 + number, 15 if name == "graph" else 31)
        model.fit(x.loc[fit, columns], y.loc[fit])
        oof.loc[valid, name] = model.predict_proba(x.loc[valid, columns])[:, 1]
        print(f"fold={fold + 1} {name} AUC={roc_auc_score(y.loc[valid], oof.loc[valid, name]):.6f}")

valid = oof.notna().all(axis=1)
blend_y, blend_oof = y.loc[valid], oof.loc[valid]
best_score, best_weights = -1.0, None
for all_weight in np.arange(0, 0.81, 0.1):
    for tab_weight in np.arange(0, 1.01 - all_weight, 0.1):
        weights = (all_weight, tab_weight, 1 - all_weight - tab_weight)
        prediction = sum(weight * blend_oof[name] for weight, name in zip(weights, sets))
        score = roc_auc_score(blend_y, prediction)
        if score > best_score:
            best_score, best_weights = score, weights
print(f"OOF AUC={best_score:.6f}; weights={best_weights}")

fold=1 all AUC=0.988030
fold=1 tabular AUC=0.987566
fold=1 graph AUC=0.500000
fold=2 all AUC=0.856984
fold=2 tabular AUC=0.829170
fold=2 graph AUC=0.500000
fold=3 all AUC=0.997294
fold=3 tabular AUC=0.997218
fold=3 graph AUC=0.500000
OOF AUC=0.969942; weights=(0.8, 0.2, -5.551115123125783e-17)


## 7. High-confidence pseudo-labelling

The training file contains many unknown labels. A base model trained on genuine labels from the early period predicts only unknown rows from that same historical period. Predictions with confidence at least 0.98 are converted to pseudo-labels. This is conservative and avoids using future test rows or validation labels.

Pseudo-labelling uses the large unknown part of the training file without pretending that unknown means licit. A model trained on genuine early labels predicts only early unknown rows. A strict confidence threshold of 0.98 is used because uncertain pseudo-labels can amplify model errors. No test rows and no validation labels are used to create pseudo-labels.

In [9]:
# Add only high-confidence pseudo-labels from early unknown historical rows.
pseudo_cutoff = train.loc[labelled, "time_step"].quantile(0.75)
pseudo_source = train.index[
    (~labelled) & (train["time_step"] <= pseudo_cutoff)
]
base_fit = labelled & (train["time_step"] <= pseudo_cutoff)
base = make_model(pos_weight, 3000)
base.fit(train.loc[base_fit, features], train.loc[base_fit, "label"])
pseudo_prob = base.predict_proba(train.loc[pseudo_source, features])[:, 1]
keep = np.maximum(pseudo_prob, 1 - pseudo_prob) >= 0.98
pseudo_index = pseudo_source[keep]
x_final = pd.concat([train.loc[labelled], train.loc[pseudo_index]])
y_final = pd.concat([
    train.loc[labelled, "label"].astype("int8"),
    pd.Series((pseudo_prob[keep] >= 0.5).astype("int8"), index=pseudo_index),
])
print(f"Pseudo-labels added: {len(pseudo_index):,}")

Pseudo-labels added: 71,443


## 8. Fit final models and generate the submission

Each feature view is refit on all genuine labels plus the selected pseudo-labels. The selected out-of-fold blend is applied to test predictions. The submission is written with the competition-required `index,target` columns.

Each feature view is refit using all genuine labels plus accepted pseudo-labels. The final prediction is a probability-weighted average rather than a hard class decision, preserving the ranking information required by ROC-AUC. The submission template is reused so that the output has exactly the required `index,target` columns.

In [10]:
# Refit each feature view and combine its test probabilities using the selected weights.
test_prediction = pd.DataFrame(index=test.index)
for number, (name, columns) in enumerate(sets.items()):
    model = make_model(pos_weight, 4000 + number, 15 if name == "graph" else 31)
    model.fit(x_final[columns], y_final)
    test_prediction[name] = model.predict_proba(test[columns])[:, 1]
pred = sum(weight * test_prediction[name] for weight, name in zip(best_weights, sets))

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
sample["target"] = np.clip(pred, 0, 1)
sample.to_csv(OUT, index=False)
print(f"saved {OUT.resolve()} shape={sample.shape}")
print(sample.head())

saved C:\Users\regan\OneDrive\Desktop\Uni\Y4S1\BT4012\BT4012_Kaggle_Project\submission_improved_temporal_graph_ensemble.csv shape=(15329, 2)
   index    target
0      0  0.000072
1      1  0.000596
2      2  0.000178
3      3  0.000134
4      4  0.000026
